# QQQ Drawdown Detection - Data Download and Feature Engineering
------------------------------------------

### Import the modules

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta
from scipy.stats import pearsonr
print('Modules are imported.')

Modules are imported.


### Download QQQ data

In [2]:
print("Descargando datos de QQQ...")
ticker = "QQQ"
data = yf.download(ticker, start="2014-01-01", end="2024-12-31", progress=False)
print(f"QQQ: {len(data)} registros")

Descargando datos de QQQ...


C:\Users\PC-Francisco\AppData\Local\Temp\ipykernel_11008\3212409042.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start="2014-01-01", end="2024-12-31", progress=False)


QQQ: 2767 registros


### Download SPY data (for correlation analysis)

In [3]:
print("Descargando datos de SPY...")
spy_data = yf.download("SPY", start="2014-01-01", end="2024-12-31", progress=False)
print(f"SPY: {len(spy_data)} registros")

Descargando datos de SPY...


C:\Users\PC-Francisco\AppData\Local\Temp\ipykernel_11008\2421575298.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  spy_data = yf.download("SPY", start="2014-01-01", end="2024-12-31", progress=False)


SPY: 2767 registros


### Download VIX data (Volatility Index)

In [4]:
print("Descargando datos de VIX...")
vix_data = yf.download("^VIX", start="2014-01-01", end="2024-12-31", progress=False)
print(f"VIX: {len(vix_data)} registros")

Descargando datos de VIX...


C:\Users\PC-Francisco\AppData\Local\Temp\ipykernel_11008\3083979657.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  vix_data = yf.download("^VIX", start="2014-01-01", end="2024-12-31", progress=False)


VIX: 2767 registros


### Prepare DataFrames (flatten MultiIndex columns)

In [5]:
# Si data tiene MultiIndex en columnas, aplanarlo
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.droplevel(1)
if isinstance(spy_data.columns, pd.MultiIndex):
    spy_data.columns = spy_data.columns.droplevel(1)
if isinstance(vix_data.columns, pd.MultiIndex):
    vix_data.columns = vix_data.columns.droplevel(1)

print("DataFrames preparados correctamente")

DataFrames preparados correctamente


### Create main DataFrame with basic QQQ columns

In [6]:
df = data[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
print(f"DataFrame principal creado con {len(df)} registros")
df.head()

DataFrame principal creado con 2767 registros


Price,Open,High,Low,Close,Volume
Date,,,,,
2014-01-02,79.499506,79.526746,79.018235,79.245247,29177500
2014-01-03,79.245254,79.317900,78.655029,78.673187,35723700
2014-01-06,78.691354,78.782157,78.092039,78.382614,32073100
2014-01-07,78.745834,79.227098,78.600544,79.109055,25860600
2014-01-08,79.127219,79.499522,78.954688,79.281586,27197400


### Feature 1: Add VIX (Volatility Index)

In [7]:
df['VIX'] = vix_data['Close']
print("✓ Feature VIX agregado")

✓ Feature VIX agregado


### Feature 2: RSI (Relative Strength Index) 14 days

In [8]:
df['RSI_14d'] = ta.momentum.RSIIndicator(close=df['Close'], window=14).rsi()
print("✓ Feature RSI_14d agregado")

✓ Feature RSI_14d agregado


### Feature 3: MACD (Moving Average Convergence Divergence)

In [9]:
macd = ta.trend.MACD(close=df['Close'])
df['MACD'] = macd.macd()
df['MACD_Signal'] = macd.macd_signal()
df['MACD_Diff'] = macd.macd_diff()
print("✓ Features MACD, MACD_Signal, MACD_Diff agregados")

✓ Features MACD, MACD_Signal, MACD_Diff agregados


### Feature 4: Historical Volatility 30 days (annualized)

In [10]:
df['Returns'] = df['Close'].pct_change()
df['Historical_Vol_30d'] = df['Returns'].rolling(window=30).std() * np.sqrt(252)
print("✓ Feature Historical_Vol_30d agregado")

✓ Feature Historical_Vol_30d agregado


### Feature 5: Bollinger Bands Width

In [11]:
bollinger = ta.volatility.BollingerBands(close=df['Close'], window=20, window_dev=2)
df['BB_High'] = bollinger.bollinger_hband()
df['BB_Low'] = bollinger.bollinger_lband()
df['BB_Mid'] = bollinger.bollinger_mavg()
df['Bollinger_Band_Width'] = (df['BB_High'] - df['BB_Low']) / df['BB_Mid']
print("✓ Feature Bollinger_Band_Width agregado")

✓ Feature Bollinger_Band_Width agregado


### Feature 6: ATR (Average True Range) 14 days

In [12]:
df['ATR_14d'] = ta.volatility.AverageTrueRange(
    high=df['High'], 
    low=df['Low'], 
    close=df['Close'], 
    window=14
).average_true_range()
print("✓ Feature ATR_14d agregado")

✓ Feature ATR_14d agregado


### Feature 7: Distance to MA200 (percentage)

In [13]:
df['MA200'] = df['Close'].rolling(window=200).mean()
df['Distance_to_MA200'] = ((df['Close'] - df['MA200']) / df['MA200']) * 100
print("✓ Feature Distance_to_MA200 agregado")

✓ Feature Distance_to_MA200 agregado


### Feature 8: Volume Ratio (current vs 20-day average)

In [14]:
df['Volume_MA20'] = df['Volume'].rolling(window=20).mean()
df['Volume_Ratio'] = df['Volume'] / df['Volume_MA20']
print("✓ Feature Volume_Ratio agregado")

✓ Feature Volume_Ratio agregado


### Feature 9: SPY-QQQ Correlation (rolling 60 days)

In [15]:
spy_returns = spy_data['Close'].pct_change()
df['SPY_Returns'] = spy_returns

def rolling_correlation(df, window=60):
    corr = []
    for i in range(len(df)):
        if i < window - 1:
            corr.append(np.nan)
        else:
            qqq_ret = df['Returns'].iloc[i-window+1:i+1]
            spy_ret = df['SPY_Returns'].iloc[i-window+1:i+1]
            mask = ~(qqq_ret.isna() | spy_ret.isna())
            if mask.sum() > 10:
                corr.append(qqq_ret[mask].corr(spy_ret[mask]))
            else:
                corr.append(np.nan)
    return corr

df['SPY_QQQ_Correlation'] = rolling_correlation(df, window=60)
print("✓ Feature SPY_QQQ_Correlation agregado")

✓ Feature SPY_QQQ_Correlation agregado


### Feature 10: 20-day Return

In [16]:
df['Return_20d'] = df['Close'].pct_change(periods=20) * 100
print("✓ Feature Return_20d agregado")

✓ Feature Return_20d agregado


### Select final features for analysis

In [17]:
features_df = df[[
    'Open', 'High', 'Low', 'Close', 'Volume',
    'VIX',
    'RSI_14d',
    'MACD', 'MACD_Signal', 'MACD_Diff',
    'Historical_Vol_30d',
    'Bollinger_Band_Width',
    'ATR_14d',
    'Distance_to_MA200',
    'Volume_Ratio',
    'SPY_QQQ_Correlation',
    'Return_20d'
]].copy()

print(f"\n✅ Proceso de feature engineering completado!")
print(f"Total de features: {len(features_df.columns)}")


✅ Proceso de feature engineering completado!
Total de features: 17


### Save complete dataset (with NaN values)

In [18]:
features_df.to_csv("QQQ_features_2014_2024.csv", index=True)
print("Archivo guardado: QQQ_features_2014_2024.csv")

Archivo guardado: QQQ_features_2014_2024.csv


### Save clean dataset (without NaN values)

In [19]:
features_clean = features_df.dropna()
features_clean.to_csv("QQQ_features_clean_2014_2024.csv", index=True)
print(f"Archivo guardado: QQQ_features_clean_2014_2024.csv")
print(f"Registros limpios (sin NaN): {len(features_clean)}")

Archivo guardado: QQQ_features_clean_2014_2024.csv
Registros limpios (sin NaN): 2568


### Display summary statistics

In [20]:
print("="*60)
print("RESUMEN DE FEATURES")
print("="*60)
features_clean.describe()

RESUMEN DE FEATURES


Price,Open,High,Low,Close,Volume,VIX,RSI_14d,MACD,MACD_Signal,MACD_Diff,Historical_Vol_30d,Bollinger_Band_Width,ATR_14d,Distance_to_MA200,Volume_Ratio,SPY_QQQ_Correlation,Return_20d
count,2568.000000,2568.000000,2568.000000,2568.000000,2.568000e+03,2568.000000,2568.000000,2568.000000,2568.000000,2568.000000,2568.000000,2568.000000,2568.000000,2568.000000,2568.000000,2568.000000,2568.000000
mean,233.998275,235.695455,232.152126,234.047512,4.123309e+07,18.211441,56.206082,1.155281,1.146204,0.009078,0.193555,0.080426,3.913862,7.017937,1.008446,0.911482,1.525471
std,118.486051,119.321618,117.529885,118.477924,2.192667e+07,7.232303,11.579505,3.501509,3.270287,1.111402,0.100276,0.042671,2.583113,8.665664,0.375083,0.057449,5.310877
min,83.054132,85.038176,78.486677,84.312531,7.079300e+06,9.140000,19.149665,-12.517747,-11.307334,-4.701736,0.053877,0.011616,0.747933,-23.706906,0.242889,0.676524,-27.846898
25%,129.264220,129.725875,128.756044,129.302780,2.539732e+07,13.340000,47.945691,-0.236920,-0.172721,-0.382807,0.128351,0.049568,1.456976,3.549396,0.766075,0.885237,-1.146837
50%,187.475179,190.339790,186.164616,188.615631,3.607705e+07,16.265000,57.186480,1.106189,1.035246,0.036974,0.167031,0.071965,3.508180,8.450618,0.934195,0.928029,2.077836
75%,325.497140,327.645247,323.194801,326.133186,5.160350e+07,21.255000,64.589481,3.055786,2.903656,0.481408,0.243620,0.099592,5.732788,11.949749,1.168215,0.955389,4.811693
max,533.480820,536.255909,531.262810,535.281128,1.986858e+08,82.690002,85.587712,10.178226,9.241613,4.420902,0.815375,0.347502,11.406842,33.084283,3.867169,0.991124,25.405479


### Display first 5 rows

In [21]:
print("="*60)
print("PRIMERAS 5 FILAS")
print("="*60)
features_clean.head()

PRIMERAS 5 FILAS


Price,Open,High,Low,Close,Volume,VIX,RSI_14d,MACD,MACD_Signal,MACD_Diff,Historical_Vol_30d,Bollinger_Band_Width,ATR_14d,Distance_to_MA200,Volume_Ratio,SPY_QQQ_Correlation,Return_20d
Date,,,,,,,,,,,,,,,,,
2014-10-16,83.054132,85.038176,83.026577,84.312531,93165800,25.200001,28.490452,-1.390336,-0.734144,-0.656193,0.165443,0.103679,1.460259,0.290485,1.605464,0.927286,-8.248555
2014-10-17,85.423965,86.241463,84.927954,85.423965,69546700,21.990000,35.549117,-1.432359,-0.873787,-0.558572,0.170290,0.105562,1.493735,1.575220,1.175532,0.930736,-6.981437
2014-10-20,85.414798,86.774237,85.240281,86.700752,41479300,18.570000,42.563155,-1.347108,-0.968451,-0.378657,0.177110,0.104837,1.496608,3.044231,0.705395,0.932063,-4.704690
2014-10-21,87.775463,88.978752,87.555016,88.978752,53609800,16.080000,52.496298,-1.083243,-0.991409,-0.091834,0.193676,0.102514,1.552422,5.685096,0.896725,0.939104,-1.933527
2014-10-22,89.254288,89.373703,88.464346,88.519463,39178000,17.870001,50.596387,-0.900806,-0.973289,0.072483,0.192206,0.096850,1.506489,5.080846,0.653567,0.939539,-3.475536


### Check missing values by column

In [22]:
print("="*60)
print("VALORES FALTANTES POR COLUMNA")
print("="*60)
features_df.isnull().sum()

VALORES FALTANTES POR COLUMNA


Price
Open                      0
High                      0
Low                       0
Close                     0
Volume                    0
VIX                       0
RSI_14d                  13
MACD                     25
MACD_Signal              33
MACD_Diff                33
Historical_Vol_30d       30
Bollinger_Band_Width     19
ATR_14d                   0
Distance_to_MA200       199
Volume_Ratio             19
SPY_QQQ_Correlation      59
Return_20d               20
dtype: int64